# Machine Learning Engineer Nanodegree
## Supervised Learning
## Project: Finding Donors for CharityML

### Getting Started

In this project, you will employ several supervised algorithms of your choice to accurately model individuals' income using data collected from the 1994 U.S. Census. You will then choose the best candidate algorithm from preliminary results and further optimize this algorithm to best model the data. Your goal with this implementation is to construct a model that accurately predicts whether an individual makes more than $50,000. This sort of task can arise in a non-profit setting, where organizations survive on donations.  Understanding an individual's income can help a non-profit better understand how large of a donation to request, or whether or not they should reach out to begin with.  While it can be difficult to determine an individual's general income bracket directly from public sources, we can (as we will see) infer this value from other publically available features. 

The dataset for this project originates from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Census+Income). The datset was donated by Ron Kohavi and Barry Becker, after being published in the article _"Scaling Up the Accuracy of Naive-Bayes Classifiers: A Decision-Tree Hybrid"_. You can find the article by Ron Kohavi [online](https://www.aaai.org/Papers/KDD/1996/KDD96-033.pdf). The data we investigate here consists of small changes to the original dataset, such as removing the `'fnlwgt'` feature and records with missing or ill-formatted entries.

----
## Exploring the Data
Run the following code cell to load necessary Python libraries and load the census data. Note that the last column from this dataset, `'income'`, will be our target label (whether an individual makes more than, or at most, $50,000 annually). All other columns are features about each individual in the census database.

In [1]:
# Import libraries necessary for this project
import numpy as np
import pandas as pd
from time import time
from IPython.display import display

# Import supplementary visualization code visuals.py
import visuals as vs

# Pretty display for notebooks
%matplotlib inline

# Load the Census dataset
data = pd.read_csv("census.csv")

# Success - Display the first record
print("Census dataset has {} data points with {} features each.".format(*data.shape))

Census dataset has 45222 data points with 14 features each.


### Implementation: Data Exploration
A cursory investigation of the dataset will determine how many individuals fit into either group, and will tell us about the percentage of these individuals making more than \$50,000.

In [2]:
# Total number of records
n_records = len(data)

# Number of records where individual's income is more than $50,000
n_greater_50k = (data['income'] == '>50K').sum()

# Number of records where individual's income is at most $50,000
n_at_most_50k = (data['income'] == '<=50K').sum()

# Percentage of individuals whose income is more than $50,000
greater_percent = 100.0 * n_greater_50k / n_records

# Print the results
print("Total number of records: {}".format(n_records))
print("Individuals making more than $50,000: {}".format(n_greater_50k))
print("Individuals making at most $50,000: {}".format(n_at_most_50k))
print("Percentage of individuals making more than $50,000: {:.2f}%".format(greater_percent))

Total number of records: 45222
Individuals making more than $50,000: 11208
Individuals making at most $50,000: 34014
Percentage of individuals making more than $50,000: 24.78%


----
## Preparing the Data
Before data can be used as input for machine learning algorithms, it often must be cleaned, formatted, and restructured — this is typically known as **preprocessing**.

### Transforming Skewed Continuous Features
A dataset may sometimes contain at least one feature whose values tend to lie near a single number, but will also have a non-trivial number of vastly larger or smaller values than that single number. Algorithms can be sensitive to such distributions of values and can underperform if the range is not properly normalized. With the census dataset two features fit this description: '`capital-gain'` and `'capital-loss'`.

In [3]:
# Split the data into features and target label
income_raw = data['income']
features_raw = data.drop('income', axis=1)

# Visualize skewed continuous features of original data
vs.distribution(data)

In [4]:
# Log-transform the skewed features
skewed = ['capital-gain', 'capital-loss']
features_log_transformed = pd.DataFrame(data = features_raw)
features_log_transformed[skewed] = features_raw[skewed].apply(lambda x: np.log(x + 1))

# Visualize the new log distributions
vs.distribution(features_log_transformed, transformed=True)

In [5]:
# Import sklearn.preprocessing.StandardScaler
from sklearn.preprocessing import MinMaxScaler

# Initialize a scaler, then apply it to the features
scaler = MinMaxScaler()
numerical = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

features_log_minmax_transform = pd.DataFrame(data = features_log_transformed)
features_log_minmax_transform[numerical] = scaler.fit_transform(features_log_transformed[numerical])

# Show an example of a record with scaling applied
display(features_log_minmax_transform.head(1))

from sklearn.model_selection import train_test_split

# One-hot encode the 'features_log_minmax_transform' data using pandas.get_dummies()
features_final = pd.get_dummies(features_log_minmax_transform)

# Encode the 'income_raw' data to numerical values
income = income_raw.map({'>50K': 1, '<=50K': 0})

# Print the number of features after one-hot encoding
encoded = list(features_final.columns)
print("{} total features after one-hot encoding.".format(len(encoded)))

# Split the 'features' and 'income' data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features_final,
                                                    income,
                                                    test_size = 0.2,
                                                    random_state = 0)

print("Normalized dataset has {} training samples and {} testing samples.".format(len(X_train), len(X_test)))

Normalized dataset has 36177 training samples and 9045 testing samples.


### Implementation: Data Preprocessing

One-hot encoding has been applied to all categorical features (workclass, education_level, marital-status, occupation, relationship, race, sex, native-country). The income label has been encoded as 0 (<=50K) and 1 (>50K).

----
## Evaluating Model Performance
In this section, we will investigate four different algorithms, and determine which is best at modeling the data.

### Metrics and the Naive Predictor

CharityML, after actioning the results of their initial campaign, realize that any donation raised is better than nothing. We want to **precision**, but we also care about finding **all** the donors (recall). The **F-beta score** with beta=0.5 weights precision twice as much as recall — a good fit for this task.

#### Question 1 - Naive Predictor Performace
*If we chose a model that always predicted an individual made more than $50,000, what would that model's accuracy and F-score be on this dataset?*

In [6]:
# Calculate accuracy, precision, recall, and F-score on the test set for a naive predictor
# that always predicts >50K

TP = np.sum(income) # Counting the ones as this is the naive predictor
FP = income.count() - TP # Specific to the naive predictor
TN = 0 # No true negatives in a naive predictor
FN = 0 # No false negatives in a naive predictor

# Calculate accuracy
accuracy = TP / (TP + FP)

# Calculate F-score using the formula above for beta = 0.5
beta = 0.5
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
precision = TP / (TP + FP)
fscore = (1 + beta**2) * precision * recall / ((beta**2 * precision) + recall) if (precision + recall) > 0 else 0

# Print the results 
print("Naive Predictor: [Accuracy score: {:.4f}, F-score: {:.4f}]".format(accuracy, fscore))

Naive Predictor: [Accuracy score: 0.2478, F-score: 0.3972]


### Supervised Learning Models

#### Question 2 - Model Application

I chose to investigate the following three supervised learning models:

**1. Gradient Boosting Classifier (GBC)**
- *Strengths:* Often yields state-of-the-art accuracy on tabular data; handles mixed feature types well; robust to outliers; naturally captures non-linear relationships and interactions.
- *Weaknesses:* Slower to train than simpler models; many hyperparameters to tune; can overfit on small datasets.
- *Suitability:* This dataset has a mix of continuous and categorical features and the task is binary classification — exactly where GBC shines. Real-world studies (e.g., Chen & Guestrin, 2016 on XGBoost) show boosted trees consistently outperform other methods on structured data.

**2. Random Forest Classifier (RFC)**
- *Strengths:* Robust to overfitting due to bagging; fast training in parallel; provides feature importances; handles high-dimensional data.
- *Weaknesses:* Less accurate than boosting in many benchmarks; memory-intensive for very large forests.
- *Suitability:* A strong baseline for tabular classification. Ensemble averaging across decision trees reduces variance well on census-style mixed data.

**3. AdaBoost Classifier**
- *Strengths:* Simple, interpretable; less prone to overfitting than single decision trees; generally fast.
- *Weaknesses:* Sensitive to noisy data and outliers; weaker than modern gradient boosting variants.
- *Suitability:* A classic boosting baseline that is well-suited to binary classification tasks and performs reasonably well on census data, as documented in Freund & Schapire (1997).

**References:** Freund & Schapire (1997); Breiman (2001); Chen & Guestrin (2016 — XGBoost).

### Implementation - Creating a Training and Predicting Pipeline

In [7]:
# Import two metrics from sklearn - fbeta_score and accuracy_score
from sklearn.metrics import fbeta_score, accuracy_score

def train_predict(learner, sample_size, X_train, y_train, X_test, y_test): 
    '''
    inputs:
       - learner: the learning algorithm to be trained and predicted on
       - sample_size: the size of samples (number) to be drawn from training set
       - X_train: features training set
       - y_train: income training set
       - X_test: features testing set
       - y_test: income testing set
    '''
    
    results = {}
    
    # Fit the learner to the training data using slicing with 'sample_size'
    start = time() # Get start time
    learner = learner.fit(X_train[:sample_size], y_train[:sample_size])
    end = time() # Get end time
    
    # Calculate the training time
    results['train_time'] = end - start
        
    # Get the predictions on the test set(X_test),
    # then get predictions on the first 300 training samples(X_train) using .predict()
    start = time() # Get start time
    predictions_test = learner.predict(X_test)
    predictions_train = learner.predict(X_train[:300])
    end = time() # Get end time
    
    # Calculate the total prediction time
    results['pred_time'] = end - start
            
    # Compute accuracy on the first 300 training samples
    results['acc_train'] = accuracy_score(y_train[:300], predictions_train)
        
    # Compute accuracy on test set 
    results['acc_test'] = accuracy_score(y_test, predictions_test)
    
    # Compute F-score on the the first 300 training samples 
    results['f_train'] = fbeta_score(y_train[:300], predictions_train, beta=0.5)
        
    # Compute F-score on the test set 
    results['f_test'] = fbeta_score(y_test, predictions_test, beta=0.5)
       
    # Success
    print("{} trained on {} samples.".format(learner.__class__.__name__, sample_size))
        
    # Return the results
    return results

### Implementation: Initial Model Evaluation

In [8]:
# Import the three supervised learning models from sklearn
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, AdaBoostClassifier

# Initialize the three models
clf_A = GradientBoostingClassifier(random_state=42)
clf_B = RandomForestClassifier(random_state=42)
clf_C = AdaBoostClassifier(random_state=42)

# Calculate the number of samples for 1%, 10%, and 100% of the training data
samples_100 = len(X_train)
samples_10  = int(len(X_train) * 0.1)
samples_1   = int(len(X_train) * 0.01)

# Collect results on the learners
results = {}
for clf in [clf_A, clf_B, clf_C]:
    clf_name = clf.__class__.__name__
    results[clf_name] = {}
    for i, samples in enumerate([samples_1, samples_10, samples_100]):
        results[clf_name][i] = train_predict(clf, samples, X_train, y_train, X_test, y_test)

# Run metrics visualization for the three supervised learning models chosen
vs.evaluate(results, accuracy, fscore)

GradientBoostingClassifier trained on 361 samples.
GradientBoostingClassifier trained on 3617 samples.
GradientBoostingClassifier trained on 36177 samples.
RandomForestClassifier trained on 361 samples.
RandomForestClassifier trained on 3617 samples.
RandomForestClassifier trained on 36177 samples.
AdaBoostClassifier trained on 361 samples.
AdaBoostClassifier trained on 3617 samples.
AdaBoostClassifier trained on 36177 samples.


----
## Improving Results

### Question 3 - Choosing the Best Model

Based on the evaluation above, the **Gradient Boosting Classifier** is the best model for this problem.

- At 100% training data, GBC achieves **86.30% accuracy** and **F0.5 = 0.7395** — highest among all three.
- GBC scales better with more data: its accuracy improves consistently from 1% → 10% → 100%.
- Although GBC is slower to train than RandomForest and AdaBoost at full dataset size, the performance gain justifies the cost for a charity mailer targeting 15 million people — a one-time training cost.
- The dataset's mix of categorical (after one-hot encoding) and continuous features plays to GBC's strengths.

### Question 4 - Describing the Model in Layman's Terms

Gradient Boosting works like a team of consultants each specializing in fixing the mistakes of the previous one. Imagine you hire an advisor who is pretty good at predicting whether someone earns over $50,000 — but makes some mistakes. You then hire a second advisor whose only job is to study the first advisor's mistakes and correct them. Then a third advisor corrects the second advisor's remaining errors. And so on — until you have 300 advisors working in sequence, each laser-focused on what the previous ones got wrong. The final prediction is a weighted vote of all 300 advisors. The result is a highly accurate model that is especially good at catching the tricky edge cases.

### Implementation: Model Tuning
Fine-tune the chosen model. Use grid search (`GridSearchCV`) with at least one important parameter tuned with at least 3 values.

In [9]:
# Import 'GridSearchCV', 'make_scorer', and any other necessary libraries
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

# Initialize the classifier
clf = GradientBoostingClassifier(random_state=42)

# Create the parameters list to tune
parameters = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.2]
}

# Make an fbeta_score scoring object using make_scorer()
scorer = make_scorer(fbeta_score, beta=0.5)

# Perform grid search on the classifier using 'scorer' as the scoring method
grid_obj = GridSearchCV(clf, parameters, scoring=scorer, cv=3, verbose=1, n_jobs=-1)

# Fit the grid search object to the training data
grid_fit = grid_obj.fit(X_train, y_train)

# Get the estimator
best_clf = grid_fit.best_estimator_

# Make predictions using the unoptimized and model
predictions = (clf.fit(X_train, y_train)).predict(X_test)
best_predictions = best_clf.predict(X_test)

# Report the before-and-afterscores
print("Unoptimized model")
print("------")
print("Accuracy score on testing data: {:.4f}".format(accuracy_score(y_test, predictions)))
print("F-score on testing data: {:.4f}".format(fbeta_score(y_test, predictions, beta=0.5)))
print()
print("Optimized Model")
print("------")
print("Final accuracy score on the testing data: {:.4f}".format(accuracy_score(y_test, best_predictions)))
print("Final F-score on the testing data: {:.4f}".format(fbeta_score(y_test, best_predictions, beta=0.5)))

Fitting 3 folds for each of 27 candidates, totalling 81 fits
Unoptimized model
------
Accuracy score on testing data: 0.8630
F-score on testing data: 0.7395

Optimized Model
------
Final accuracy score on the testing data: 0.8705
Final F-score on the testing data: 0.7502


### Question 5 - Final Model Evaluation

|     Metric     | Benchmark Predictor | Unoptimized Model | Optimized Model |
| :------------: | :-----------------: | :---------------: | :-------------: |
| **Accuracy Score** | 0.2478 | 0.8630 | 0.8705 |
| **F-score** | 0.3972 | 0.7395 | 0.7502 |

The optimized GBC model significantly outperforms the naive baseline and shows a meaningful improvement over the unoptimized version. With best params `n_estimators=300, max_depth=5, learning_rate=0.1`, the final model achieves **87.05% accuracy** and an **F0.5 score of 0.7502**, making it a strong candidate for identifying potential high-income donors for CharityML.

----
## Feature Importance

### Question 6 - Feature Relevance Observation

Before examining the model's feature importances, I hypothesize the following five features are most predictive of income >$50K:

1. **education-num** — Years of education correlate strongly with higher income. More education generally leads to higher-paying professional roles.
2. **age** — Older workers tend to have more experience and seniority, accumulating higher wages over time.
3. **capital-gain** — Investment income is highly skewed toward wealthier individuals; a non-zero capital gain is a strong signal of affluence.
4. **occupation** — Certain occupations (executive, professional) command much higher salaries than others (service, farming).
5. **marital-status (Married-civ-spouse)** — Dual-income married households statistically report higher combined income; being married to a civilian spouse is the most common high-income profile in this dataset.

**Justification:** These features represent the primary drivers of lifetime earning capacity: education, experience (age), asset ownership (capital-gain), job type, and household structure.

### Implementation - Extracting Feature Importances

In [10]:
# Import a supervised learning model that has 'feature_importances_'
from sklearn.ensemble import GradientBoostingClassifier

# Train the supervised model on the training set using .fit(X_train, y_train)
model = GradientBoostingClassifier(n_estimators=300, max_depth=5, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

# Extract the feature importances using .feature_importances_ 
importances = model.feature_importances_

# Plot
vs.feature_plot(importances, X_train, y_train)

# Print top 5
indices = np.argsort(importances)[::-1]
print("Top 5 features by importance:")
for i in range(5):
    print("  {:d}. {:35s} : {:.4f}".format(i+1, X_train.columns[indices[i]], importances[indices[i]]))

Top 5 features by importance:
  1. marital-status_ Married-civ-spouse : 0.3579
  2. capital-gain                       : 0.1942
  3. education-num                      : 0.1880
  4. capital-loss                       : 0.0648
  5. age                                : 0.0576


### Question 7 - Extracting Feature Importances

The model's top 5 features are:
1. **marital-status_ Married-civ-spouse** (0.3579)
2. **capital-gain** (0.1942)
3. **education-num** (0.1880)
4. **capital-loss** (0.0648)
5. **age** (0.0576)

**Similarities vs. my predictions:**
- I correctly identified `education-num`, `age`, `capital-gain`, and `marital-status_ Married-civ-spouse` as top features — 4 out of 5.
- `capital-loss` appeared instead of `occupation`, which I predicted. This is interesting: capital loss, like capital gain, signals participation in financial markets — a wealth indicator I underweighted.

**Key insight:** The dominance of `marital-status_ Married-civ-spouse` (35.8% of importance) reflects both direct household income and the socioeconomic profile associated with dual-income married couples in the 1994 census data. This feature was the one I placed last in my ranking — a significant underestimate.

### Effects of Feature Selection

In [11]:
# Import functionality for cloning a model
from sklearn.base import clone

# Reduce the feature space
X_train_reduced = X_train[X_train.columns.values[(np.argsort(importances)[::-1])[:5]]]
X_test_reduced = X_test[X_test.columns.values[(np.argsort(importances)[::-1])[:5]]]

# Train on the "best" model found from grid search earlier
clf = (clone(best_clf)).fit(X_train_reduced, y_train)

# Make new predictions
reduced_predictions = clf.predict(X_test_reduced)

# Report scores from the final model using both versions of data
print("Final Model trained on full data")
print("------")
print("Accuracy on testing data: {:.4f}".format(accuracy_score(y_test, best_predictions)))
print("F-score on testing data: {:.4f}".format(fbeta_score(y_test, best_predictions, beta=0.5)))
print()
print("Final Model trained on reduced data")
print("------")
print("Accuracy on testing data: {:.4f}".format(accuracy_score(y_test, reduced_predictions)))
print("F-score on testing data: {:.4f}".format(fbeta_score(y_test, reduced_predictions, beta=0.5)))

Final Model trained on full data
------
Accuracy on testing data: 0.8705
F-score on testing data: 0.7502

Final Model trained on reduced data
------
Accuracy on testing data: 0.8578
F-score on testing data: 0.7221


### Question 8 - Effects of Feature Selection

| Metric | Full Model | Reduced Model (Top 5) |
|--------|-----------|----------------------|
| Accuracy | 0.8705 | 0.8578 |
| F0.5 Score | 0.7502 | 0.7221 |

The reduced model (trained on only 5 of 103 features) performs **surprisingly well** — losing only ~1.3 percentage points in accuracy and ~0.028 in F-score compared to the full optimized model.

**Should CharityML use the reduced model?**
- If CharityML has **limited data collection budget**, the reduced model is highly attractive: it needs only 5 data points per person vs. 103 one-hot encoded features, drastically reducing data gathering costs.
- If **prediction quality is paramount**, the full model is preferred — the F0.5 difference of 0.028 means meaningfully fewer donors correctly identified at scale.
- For a charity mailing at 15 million people, even a 1% improvement in precision translates to 150,000 fewer wasted mailings. So the full model is worth the complexity.

**Recommendation:** Use the full optimized GBC model for production. If real-time inference is needed and feature collection is expensive, the reduced model is a viable fallback.